# Audiobook Enhancer — Colab Bootstrap

This notebook is a bootstrapper only. It never contains pipeline logic —
all processing code lives in the `audiobook_engine` package in the repo.
Edit code there, not here. See `02-Architecture.md` in the vault.

In [ ]:
# 1. Clone or pull the repo
REPO_URL = "https://github.com/hasnaintypes/audiobook-enhancer.git"
!test -d audiobook-enhancer && (cd audiobook-enhancer && git pull) || git clone "$REPO_URL" audiobook-enhancer

In [ ]:
# 2. Install pinned dependencies
%cd audiobook-enhancer
!pip install -q -r config/requirements.lock
!pip install -q -e .

In [ ]:
# 3. Load Neon Object Storage credentials from Colab's Secrets manager
# (the key icon in the left sidebar) — NOT pasted into this cell. Add each
# of these as a secret there first, with "Notebook access" enabled:
#   AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_ENDPOINT_URL_S3, AWS_REGION
# Values come from audiobook-enhancer/.env.local on your own machine
# (written there by `neon deploy` — see docs/07-Decisions.md D007).
import os
from google.colab import userdata

for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_ENDPOINT_URL_S3", "AWS_REGION"]:
    os.environ[key] = userdata.get(key)

print("Neon Object Storage credentials loaded from Colab Secrets.")

In [ ]:
# 4. Sanity check: run the fast unit test suite here to confirm the clone +
# dependency install actually produced a working environment.
!pytest tests/unit -q

In [ ]:
# 5. Example: run the full pipeline (extract -> analyze -> upload) against
# real sample URLs, writing originals to the Neon Object Storage bucket.
script = '''
from pathlib import Path
from audiobook_engine.ingestion.youtube import extract_audio, ExtractionError
from audiobook_engine.analysis.io import load_mono
from audiobook_engine.analysis.quality import analyze_quality
from audiobook_engine.analysis.strategy import select_strategy
from audiobook_engine.jobstore.storage import upload_file, object_key, presigned_url

URLS = [
    # Replace with the URL(s) you want to process
]
OUTPUT_DIR = Path("/content/samples")

for i, url in enumerate(URLS, start=1):
    job_id = f"sample-{i:03d}"
    print("=" * 80)
    print(url, "->", job_id)
    try:
        extracted = extract_audio(url, OUTPUT_DIR)
        print(f"Title: {extracted.title}")

        samples, sr = load_mono(extracted.audio_path)
        report = analyze_quality(samples, sr)
        decision = select_strategy(report)
        print(report)
        print(decision)

        key = object_key(job_id, "original", extracted.audio_path.name)
        upload_file(extracted.audio_path, key)
        print(f"Uploaded to Neon Object Storage: {key}")

        url_out = presigned_url(key, expires_in=600)
        print(f"Presigned URL (10 min): {url_out[:80]}...")
    except ExtractionError as e:
        print(f"Extraction failed [{e.kind.value}]: {e}")
'''
with open("/content/run_job_example.py", "w") as f:
    f.write(script)

!python /content/run_job_example.py